# 2.4 Element IDs and full-text deduplication

*Author: La Anh Hào · code: `src/13_dedup_export.ipynb` (`parse_paper`, `_ElementStore`, `check_paper`; 24
offline tests)*

The statement asks for one `elements` row per distinct element of a paper over all of its versions: when the
cleaned text of an element is identical in two versions, both versions point to the same `element_id`, and
the `hierarchy` table records where it sits in each version. `parse_paper` gives the IDs while it walks the
tree of each version, in document order, starting with version 1.

## 2.4.1 Element IDs

`element_id` is `<paper folder>#<prefix><n>`, for example `2609-32834#s12`: the folder name of the paper (the
ID with a hyphen), then a prefix for the kind and a number counted per prefix in order of first appearance.

| Kind | Prefix | Kind | Prefix | Kind | Prefix |
|---|---|---|---|---|---|
| `document` | `doc` | `subsubsection` | `sssec` | `item` | `item` |
| `chapter` | `ch` | `paragraph` | `par` | `sentence` | `s` |
| `section` | `sec` | `abstract` | `abs` | `formula` | `eq` |
| `subsection` | `ssec` | `itemize` | `list` | `figure` | `fig` |

The document root is `<paper folder>#doc` in every version, without a number.

## 2.4.2 When two nodes are the same element

- A **leaf** (`sentence`, `formula`, `figure`) is identified by `(kind, content)`. The same cleaned sentence
  gets the same ID in any version and under any parent; a sentence written twice in one version is one
  element with two `hierarchy` rows.
- An **inner node** is identified by `(kind, content, parent element_id, k)`. The parent keeps apart two
  subsections titled *Setup* in two different sections: with the title alone, their children would be mixed.
  `k` counts the earlier siblings with the same kind and content under that parent in the version, which
  keeps apart the two untitled lists of one section.
- The match is **exact** on the text produced by chapter [2.2](normalization.md), as the statement asks: a
  sentence in which one word changed is a new element. The normalization is what makes exact matching
  useful: the same sentence written with a macro in v1 and its expansion in v2, `\[...\]` in one version and
  `$$...$$` in the other, or a different `\cite` key for the same work, is identical once cleaned.
- The citation keys are renamed (chapter [2.3](references.md)) **before** the tree is built, so the sentence
  that cites `vaswani2017` in v1 and `vit_transformer` in v2 is one element.

## 2.4.3 Checks

Before any ID is given, the tree of a version is checked (one `document` root, known kinds, leaves without
children); a malformed tree makes the version `error` without leaving half of its rows behind. After a paper
is parsed, `check_paper` verifies the invariants of its rows: IDs well formed and unique, every `element_id`
and `parent_id` of `hierarchy` present in `elements` and never null, `<paper folder>#doc` the only parent
that is never a child in each version, every version of `metadata.json` listed once, and no rows for a
version that is not `ok`. `15_run_parsing` checks the same over the final Parquet tables; the full run
reports no broken invariant (chapter [2.5](running.md)).

## 2.4.4 A worked example

The paper below has two versions. Its file is taken from the output of the full run (`data/parsed/<id>.json`),
and everything in this section is computed from it when the book is built.

In [ ]:
import pandas as pd

import report_data as rd

rd.check()
ex = rd.example()
elements = ex["elements"]                     # element_id -> [kind, content]
order = {eid: i for i, eid in enumerate(elements)}


def nodes(tree, parent=None, depth=0):
    # (depth, element_id, parent_id) of every node of a tree written as [element_id, child, child, ...]
    eid, *children = tree
    yield depth, eid, parent
    for child in children:
        yield from nodes(child, eid, depth + 1)


versions = {int(v): list(nodes(tree)) for v, tree in ex["trees"].items()}
first, last = min(versions), max(versions)
ids = {v: {eid for _, eid, _ in rows} for v, rows in versions.items()}
title = elements[f"{ex['folder_id']}#doc"][1]
final = rd.stats()["parser_version"]
note = ("" if ex["parser_version"] == final else
        f" The file was written by parser `{ex['parser_version'].split('|')[0]}` and kept by "
        f"`{final.split('|')[0]}`, whose output for this paper is the same (chapter [2.5](running.md)).")
rd.md(f"**{ex['arxiv_id']}** — *{title}*.{note}")
t = pd.DataFrame([{"Version": v, "Status": r["status"], "Tree nodes": len(versions.get(v, [])),
                   "Distinct elements": len(ids.get(v, set())), "Source of the references": r["bib_source"]}
                  for v, r in ((r["version"], r) for r in ex["versions"])])
rd.table(t, "Table 2.4 – The versions of the example paper")
total_nodes = sum(len(rows) for rows in versions.values())
rd.md(f"The {len(versions)} trees have **{total_nodes}** nodes in all; the paper has **{len(elements)}** rows in "
      f"`elements` and **{ex['hierarchy_rows']}** in `hierarchy` (one per node except the document root of each "
      f"version).")

**Structure.** The headings and lists of both versions, with their element IDs. A heading whose title did
not change keeps its ID; a heading whose title changed, even only in capital letters, is a new element.

In [ ]:
inner = [eid for eid in elements if elements[eid][0] not in rd.LEAVES + ("item",)]
depth = {}
for v in sorted(versions):
    for d, eid, _ in versions[v]:
        depth.setdefault(eid, d)
rows = []
for eid in inner:
    kind, content = elements[eid]
    rows.append({"element_id": f"`{eid.split('#')[1]}`",
                 "Kind": kind, "Title": (" " * depth.get(eid, 0)) + (content or "–"),
                 **{f"v{v}": "✓" if eid in ids[v] else "" for v in sorted(versions)}})
rd.table(pd.DataFrame(rows), "Table 2.5 – Inner nodes of the example (indented by depth; ✓: present in the version)")

In [ ]:
def leaf_counts():
    out = []
    for kind in rd.LEAVES:
        of_kind = {eid for eid in elements if elements[eid][0] == kind}
        both = of_kind & ids[first] & ids[last]
        out.append({"Kind": kind, f"v{first} only": len((of_kind & ids[first]) - ids[last]),
                    f"v{last} only": len((of_kind & ids[last]) - ids[first]), "Both versions": len(both),
                    "Elements": len(of_kind)})
    return pd.DataFrame(out)


counts = leaf_counts()
rd.table(counts, f"Table 2.6 – Leaves of v{first} and v{last}: shared and specific elements")
shared = int(counts["Both versions"].sum())
rd.md(f"**{shared}** leaves are identical in v{first} and v{last} and are stored once; the "
      f"{int(counts[f'v{first} only'].sum())} leaves of v{first} only and {int(counts[f'v{last} only'].sum())} of "
      f"v{last} only are the sentences, formulas and figures that the authors changed.")

**One element, two parents.** When a section is renamed, its sentences that did not change are still the
same elements: they hang under the old heading in one version and under the new one in the other. This is the
case of the worked example of the statement, where one sentence has a `hierarchy` row in each version with a
different parent.

In [ ]:
parent = {v: {eid: p for _, eid, p in rows if p} for v, rows in versions.items()}
moved = [eid for eid in elements if elements[eid][0] in rd.LEAVES
         and eid in parent[first] and eid in parent[last] and parent[first][eid] != parent[last][eid]]
if moved:
    # the sentence with the fewest LaTeX commands reads best
    eid = min(moved, key=lambda e: (elements[e][0] != "sentence", elements[e][1].count("\\"), abs(len(elements[e][1]) - 160)))
    rd.md(f"Element `{eid}` ({elements[eid][0]}): *{elements[eid][1][:300]}*")
    rows = [{"arxiv_id": ex["arxiv_id"], "version": v, "element_id": eid, "parent_id": parent[v][eid],
             "parent": elements[parent[v][eid]][1] or elements[parent[v][eid]][0]} for v in (first, last)]
    rd.table(pd.DataFrame(rows), "Table 2.7 – Its rows in hierarchy (the last column is the parent's title, for reading)")
    rd.md(f"{len(moved)} leaves of this paper have a different parent in v{first} and v{last}.")
else:
    rd.md("In this paper no leaf changed parent between the versions.")

**References.** The BibTeX rows of the paper after deduplication over both versions (the first five):

In [ ]:
bib = pd.DataFrame(ex["bibtex"])
cols = ["citation_key", "entry_type", "title", "authors", "year", "doi", "journal"]
bib = bib[[c for c in cols if c in bib.columns]].head(5).copy()
if "authors" in bib:
    bib["authors"] = bib["authors"].map(lambda a: ", ".join(a[:3]) + (" et al." if a and len(a) > 3 else "") if a else "")
rd.table(bib, f"Table 2.8 – The first BibTeX rows of the example ({len(ex['bibtex'])} rows in all)", {"year": "{:.0f}"})